# Tanimoto-Binned Analysis

Loads all five method score CSVs and analyses results stratified by Tanimoto similarity bins.
No fixed τ_t cutoff is applied — instead we examine how variant quality varies across the full similarity spectrum.

**Tanimoto bins:** `[0, 0.3)` · `[0.3, 0.5)` · `[0.5, 0.7)` · `[0.7, 1.0]`

**Quality metric:** desirability ≥ 0.5 (physicochemical gate only)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

ROOT = Path('.').resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

GEN_DIR = ROOT / 'data' / 'generation_stratified'

METHODS = {
    'baseline':  GEN_DIR / 'baseline'  / 'baseline_scores.csv',
    'crem':      GEN_DIR / 'crem'      / 'crem_scores.csv',
    'libinvent': GEN_DIR / 'libinvent' / 'libinvent_scores.csv',
    'mmpdb':     GEN_DIR / 'mmpdb'     / 'mmpdb_scores.csv',
    'jtvae':     GEN_DIR / 'jtvae'     / 'jtvae_scores.csv',
}

METHOD_COLORS = {
    'baseline':  '#888888',
    'crem':      '#2196F3',
    'libinvent': '#FF9800',
    'mmpdb':     '#4CAF50',
    'jtvae':     '#9C27B0',
}

TAN_BINS   = [0.0, 0.3, 0.5, 0.7, 1.001]   # right edge slightly > 1 to include 1.0
TAN_LABELS = ['<0.3', '0.3-0.5', '0.5-0.7', '≥0.7']
TAU_D      = 0.5
QUAL_BINS  = ['<0.5', '0.5-0.7', '0.7-0.85', '0.85-1.0']

# Load
dfs = {}
for name, path in METHODS.items():
    if path.exists():
        dfs[name] = pd.read_csv(path)
        print(f'{name:<12}: {len(dfs[name]):>6,} rows')
    else:
        print(f'{name:<12}: MISSING — {path}')

df_all = pd.concat(dfs.values(), ignore_index=True)
df_all['tan_bin'] = pd.cut(df_all['tanimoto'], bins=TAN_BINS, labels=TAN_LABELS, right=False)
df_all['des_pass'] = df_all['desirability'] >= TAU_D
print(f'\nTotal rows: {len(df_all):,}')

---
## 1 — Variant counts by method × tanimoto bin

In [ ]:
count_tbl = (
    df_all
    .groupby(['method', 'tan_bin'], observed=True)
    .size()
    .unstack('tan_bin', fill_value=0)
    .reindex(columns=TAN_LABELS)
)
count_tbl['Total'] = count_tbl.sum(axis=1)
print('Variant counts by method × tanimoto bin')
print(count_tbl.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

x      = np.arange(len(TAN_LABELS))
names  = list(dfs.keys())
n      = len(names)
width  = 0.15
offsets= np.linspace(-(n-1)/2, (n-1)/2, n) * width

for i, name in enumerate(names):
    vals = [count_tbl.loc[name, b] if name in count_tbl.index else 0 for b in TAN_LABELS]
    ax.bar(x + offsets[i], vals, width=width, label=name, color=METHOD_COLORS[name])

ax.set_xticks(x)
ax.set_xticklabels(TAN_LABELS)
ax.set_xlabel('Tanimoto bin')
ax.set_ylabel('Variant count')
ax.set_title('Variant distribution across Tanimoto bins')
ax.legend()
plt.tight_layout()
plt.savefig(GEN_DIR / 'analysis_count_by_tanbin.png', dpi=150)
plt.show()

---
## 2 — Desirability pass rate by method × tanimoto bin

For each (method, tanimoto bin), what fraction of variants pass desirability ≥ 0.5?

In [ ]:
des_tbl = (
    df_all
    .groupby(['method', 'tan_bin'], observed=True)['des_pass']
    .mean()
    .mul(100)
    .unstack('tan_bin')
    .reindex(columns=TAN_LABELS)
    .round(1)
)
print('Desirability pass rate (%) by method × tanimoto bin')
print(des_tbl.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for i, name in enumerate(names):
    if name not in des_tbl.index:
        continue
    vals = des_tbl.loc[name, TAN_LABELS].values.astype(float)
    ax.plot(TAN_LABELS, vals, marker='o', label=name,
            color=METHOD_COLORS[name], linewidth=2)

ax.set_xlabel('Tanimoto bin')
ax.set_ylabel('Desirability pass rate (%)')
ax.set_title('Desirability pass rate across Tanimoto bins')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(GEN_DIR / 'analysis_despass_by_tanbin.png', dpi=150)
plt.show()

---
## 3 — Tanimoto distribution per method (violin / KDE)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for i, name in enumerate(names):
    if name not in dfs:
        continue
    vals = dfs[name]['tanimoto'].dropna()
    vals.plot.kde(ax=ax, label=name, color=METHOD_COLORS[name], linewidth=2)

for t in [0.3, 0.5, 0.7]:
    ax.axvline(t, color='gray', linestyle='--', linewidth=0.8)

ax.set_xlabel('Tanimoto similarity to spec')
ax.set_ylabel('Density')
ax.set_title('Tanimoto distribution by method')
ax.legend()
ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(GEN_DIR / 'analysis_tanimoto_kde.png', dpi=150)
plt.show()

print('Mean tanimoto per method:')
for name in names:
    if name in dfs:
        print(f'  {name:<12}: {dfs[name]["tanimoto"].mean():.3f}')

---
## 4 — Hit count by method × tanimoto bin

Here a **hit** = desirability ≥ 0.5 only (no tanimoto threshold applied).
This shows where each method's useful variants concentrate in similarity space.

In [ ]:
hits_count = (
    df_all[df_all['des_pass']]
    .groupby(['method', 'tan_bin'], observed=True)
    .size()
    .unstack('tan_bin', fill_value=0)
    .reindex(columns=TAN_LABELS)
)
hits_count['Total'] = hits_count.sum(axis=1)
print('Hit counts (des >= 0.5) by method × tanimoto bin')
print(hits_count.to_string())

print('\nHit rate (hits / all variants in that bin):')
hit_rate = (
    df_all
    .groupby(['method', 'tan_bin'], observed=True)['des_pass']
    .agg(['sum', 'count'])
    .assign(rate=lambda d: d['sum'] / d['count'] * 100)
    ['rate']
    .unstack('tan_bin')
    .reindex(columns=TAN_LABELS)
    .round(1)
)
print(hit_rate.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: absolute hit counts
for i, name in enumerate(names):
    if name not in hits_count.index:
        continue
    vals = [hits_count.loc[name, b] for b in TAN_LABELS]
    axes[0].bar(x + offsets[i], vals, width=width, label=name, color=METHOD_COLORS[name])
axes[0].set_xticks(x)
axes[0].set_xticklabels(TAN_LABELS)
axes[0].set_xlabel('Tanimoto bin')
axes[0].set_ylabel('Hit count')
axes[0].set_title('Hit count (des ≥ 0.5) per tanimoto bin')
axes[0].legend(fontsize=8)

# Right: hit rate
for i, name in enumerate(names):
    if name not in hit_rate.index:
        continue
    vals = hit_rate.loc[name, TAN_LABELS].values.astype(float)
    axes[1].plot(TAN_LABELS, vals, marker='o', label=name,
                 color=METHOD_COLORS[name], linewidth=2)
axes[1].set_xlabel('Tanimoto bin')
axes[1].set_ylabel('Hit rate (%)')
axes[1].set_title('Hit rate (des ≥ 0.5) per tanimoto bin')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(GEN_DIR / 'analysis_hits_by_tanbin.png', dpi=150)
plt.show()

---
## 5 — Seed quality bin × tanimoto bin heatmap

Does starting seed quality influence where variants land in tanimoto space?

In [ ]:
fig, axes = plt.subplots(1, len(dfs), figsize=(4 * len(dfs), 4))
if len(dfs) == 1:
    axes = [axes]

for ax, (name, df) in zip(axes, dfs.items()):
    pivot = (
        df.assign(tan_bin=pd.cut(df['tanimoto'], bins=TAN_BINS, labels=TAN_LABELS, right=False))
        .groupby(['quality_bin', 'tan_bin'], observed=True)
        .size()
        .unstack('tan_bin', fill_value=0)
        .reindex(index=QUAL_BINS, columns=TAN_LABELS, fill_value=0)
    )
    sns.heatmap(
        pivot, ax=ax, annot=True, fmt='d', cmap='Blues',
        cbar=False, linewidths=0.5
    )
    ax.set_title(name)
    ax.set_xlabel('Tanimoto bin')
    ax.set_ylabel('Seed quality bin' if ax == axes[0] else '')
    if ax != axes[0]:
        ax.set_ylabel('')
        ax.set_yticklabels([])

plt.suptitle('Variant counts: seed quality bin × tanimoto bin', y=1.02)
plt.tight_layout()
plt.savefig(GEN_DIR / 'analysis_qualbin_tanbin_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6 — Desirability component breakdown per tanimoto bin

Which physicochemical component fails most often in each tanimoto range?

In [ ]:
COMP_COLS = ['mw_score', 'clogp_score', 'tpsa_score', 'hbd_score', 'hba_score', 'rot_bonds_score']

# Fraction of variants where each component = 0, by tanimoto bin
comp_fail = (
    df_all
    .groupby('tan_bin', observed=True)[COMP_COLS]
    .apply(lambda g: (g == 0).mean() * 100)
    .reindex(TAN_LABELS)
)
print('Component zero-rate (%) by tanimoto bin (all methods combined):')
print(comp_fail.round(1).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

comp_labels = [c.replace('_score', '') for c in COMP_COLS]
comp_colors = plt.cm.Set2(np.linspace(0, 1, len(COMP_COLS)))

for col, label, color in zip(COMP_COLS, comp_labels, comp_colors):
    ax.plot(TAN_LABELS, comp_fail[col].values, marker='o',
            label=label, color=color, linewidth=2)

ax.set_xlabel('Tanimoto bin')
ax.set_ylabel('Component failure rate (%)')
ax.set_title('Physicochemical component failure rate by tanimoto bin')
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(GEN_DIR / 'analysis_component_fail_by_tanbin.png', dpi=150)
plt.show()

---
## 7 — Summary table

In [ ]:
rows = []
for name, df in dfs.items():
    df = df.copy()
    df['tan_bin'] = pd.cut(df['tanimoto'], bins=TAN_BINS, labels=TAN_LABELS, right=False)
    for tb in TAN_LABELS:
        sub = df[df['tan_bin'] == tb]
        if len(sub) == 0:
            continue
        hits = (sub['desirability'] >= TAU_D).sum()
        rows.append({
            'method':      name,
            'tan_bin':     tb,
            'n_variants':  len(sub),
            'mean_tan':    sub['tanimoto'].mean(),
            'mean_des':    sub['desirability'].mean(),
            'des_pass_pct': hits / len(sub) * 100,
            'n_hits':      hits,
        })

summary = pd.DataFrame(rows)
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_rows', 100)
print(summary.to_string(index=False))